In [4]:
import torchaudio
from pathlib import Path
import zipfile
import librosa
import torch
import numpy as np
from SMT import *
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from tqdm import tqdm
import json
from torch.utils.data import Dataset
import re

In [5]:
librispeech_ds = torchaudio.datasets.LIBRISPEECH(
    root=str("./data/"), url="dev-clean", download=True,
)

In [6]:
kmeans, norm_beta, norm_patches, utterance_bounds = calc_smt_on_librispeech(librispeech_ds, patch_length=50, num_utterances=500)

calc mels
patch utterances
apply kmeans
spectral decomp


In [7]:
norm_beta.shape

(355521, 128)

In [8]:
texts = [re.sub(r"[^a-z0-9\s]", "", librispeech_ds.get_metadata(idx)[2].lower()) for idx in range(len(utterance_bounds))]

In [9]:
len(texts)

500

In [10]:
import math
from dataclasses import dataclass
from typing import List, Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


# -----------------------------
# 1. Model
# -----------------------------
class SMTCTCLinear(nn.Module):
    def __init__(
        self,
        d_in: int,
        vocab_size: int,
        dropout: float = 0.1,
        use_layernorm: bool = True,
    ):
        super().__init__()
        self.ln = nn.LayerNorm(d_in) if use_layernorm else nn.Identity()
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.proj = nn.Linear(d_in, vocab_size)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """
        z: (B, T, D)
        returns logits: (B, T, V)
        """
        z = self.ln(z)
        z = self.drop(z)
        logits = self.proj(z)
        return logits


# -----------------------------
# 2. Tokenizer
# -----------------------------
class CharTokenizer:
    """
    Very simple character tokenizer for CTC.
    Index 0 is reserved for blank.
    """
    def __init__(self, alphabet: str = "abcdefghijklmnopqrstuvwxyz '"):
        self.blank_id = 0
        self.alphabet = alphabet
        self.char_to_id = {ch: i + 1 for i, ch in enumerate(alphabet)}
        self.id_to_char = {i + 1: ch for i, ch in enumerate(alphabet)}
        self.vocab_size = len(alphabet) + 1  # + blank

    def encode(self, text: str) -> List[int]:
        text = text.lower()
        ids = []
        for ch in text:
            if ch in self.char_to_id:
                ids.append(self.char_to_id[ch])
        return ids

    def decode(self, ids: List[int]) -> str:
        return "".join(self.id_to_char[i] for i in ids if i in self.id_to_char)


# -----------------------------
# 3. Collate function
# -----------------------------
def ctc_collate_fn(batch: List[Dict], tokenizer: CharTokenizer) -> Dict[str, torch.Tensor]:
    """
    Expected per-sample format:
    {
        "features": FloatTensor (T, D),   # SMT embedding sequence
        "text": str
    }
    """
    batch_size = len(batch)
    feat_lens = [item["features"].shape[0] for item in batch]
    d_in = batch[0]["features"].shape[1]
    max_t = max(feat_lens)

    feats = torch.zeros(batch_size, max_t, d_in, dtype=torch.float32)
    feat_lens_tensor = torch.tensor(feat_lens, dtype=torch.long)

    all_targets = []
    target_lens = []

    for i, item in enumerate(batch):
        x = item["features"]
        t = x.shape[0]
        feats[i, :t] = x

        target_ids = tokenizer.encode(item["text"])
        all_targets.extend(target_ids)
        target_lens.append(len(target_ids))

    targets = torch.tensor(all_targets, dtype=torch.long)
    target_lens = torch.tensor(target_lens, dtype=torch.long)

    return {
        "features": feats,               # (B, T, D)
        "feature_lens": feat_lens_tensor,
        "targets": targets,              # (sum target lengths,)
        "target_lens": target_lens,      # (B,)
        "texts": [item["text"] for item in batch],
    }


# -----------------------------
# 4. Greedy CTC decoding
# -----------------------------
def ctc_greedy_decode(
    log_probs: torch.Tensor,
    input_lens: torch.Tensor,
    blank_id: int,
) -> List[List[int]]:
    """
    log_probs: (T, B, V)
    input_lens: (B,)
    returns: list of token-id sequences after CTC collapse
    """
    pred_ids = log_probs.argmax(dim=-1)  # (T, B)
    pred_ids = pred_ids.transpose(0, 1)  # (B, T)

    decoded = []
    for b in range(pred_ids.shape[0]):
        seq = pred_ids[b, :input_lens[b]].tolist()
        collapsed = []
        prev = None
        for tok in seq:
            if tok != prev and tok != blank_id:
                collapsed.append(tok)
            prev = tok
        decoded.append(collapsed)
    return decoded


# -----------------------------
# 5. Error rate utilities
# -----------------------------
def levenshtein_distance(ref: List[str], hyp: List[str]) -> int:
    """
    Token-level Levenshtein distance.
    """
    m, n = len(ref), len(hyp)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,      # deletion
                dp[i][j - 1] + 1,      # insertion
                dp[i - 1][j - 1] + cost,  # substitution
            )
    return dp[m][n]


def char_error_rate(ref: str, hyp: str) -> float:
    ref_chars = list(ref)
    hyp_chars = list(hyp)
    if len(ref_chars) == 0:
        return 0.0 if len(hyp_chars) == 0 else 1.0
    return levenshtein_distance(ref_chars, hyp_chars) / len(ref_chars)


def word_error_rate(ref: str, hyp: str) -> float:
    ref_words = ref.split()
    hyp_words = hyp.split()
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else 1.0
    return levenshtein_distance(ref_words, hyp_words) / len(ref_words)


# -----------------------------
# 6. Train / eval step
# -----------------------------
@dataclass
class EpochStats:
    loss: float
    cer: float
    wer: float


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    ctc_loss_fn: nn.CTCLoss,
    tokenizer: CharTokenizer,
    device: torch.device,
    optimizer: torch.optim.Optimizer = None,
    grad_clip_norm: float = 5.0,
) -> EpochStats:
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_batches = 0
    total_cer = 0.0
    total_wer = 0.0
    total_examples = 0

    for batch in loader:
        feats = batch["features"].to(device)                 # (B, T, D)
        feat_lens = batch["feature_lens"]                    # keep on CPU for CTCLoss compatibility
        targets = batch["targets"]                           # keep on CPU for CTCLoss compatibility
        target_lens = batch["target_lens"]                   # keep on CPU for CTCLoss compatibility
        texts = batch["texts"]

        logits = model(feats)                                # (B, T, V)
        log_probs = F.log_softmax(logits, dim=-1)            # (B, T, V)
        log_probs_t = log_probs.transpose(0, 1)              # (T, B, V)

        loss = ctc_loss_fn(
            log_probs_t,
            targets,
            feat_lens,
            target_lens,
        )

        if training:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            if grad_clip_norm is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
            optimizer.step()

        with torch.no_grad():
            pred_token_ids = ctc_greedy_decode(
                log_probs_t.detach().cpu(),
                feat_lens.cpu(),
                blank_id=tokenizer.blank_id,
            )
            pred_texts = [tokenizer.decode(ids) for ids in pred_token_ids]

            batch_cer = 0.0
            batch_wer = 0.0
            for ref, hyp in zip(texts, pred_texts):
                ref_norm = ref.lower()
                batch_cer += char_error_rate(ref_norm, hyp)
                batch_wer += word_error_rate(ref_norm, hyp)

            bsz = len(texts)
            total_cer += batch_cer
            total_wer += batch_wer
            total_examples += bsz

        total_loss += loss.item()
        total_batches += 1

    return EpochStats(
        loss=total_loss / max(total_batches, 1),
        cer=total_cer / max(total_examples, 1),
        wer=total_wer / max(total_examples, 1),
    )


# -----------------------------
# 7. Full training loop
# -----------------------------
def train_ctc_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    tokenizer: CharTokenizer,
    device: torch.device,
    num_epochs: int = 20,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    grad_clip_norm: float = 5.0,
):
    model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    ctc_loss_fn = nn.CTCLoss(
        blank=tokenizer.blank_id,
        reduction="mean",
        zero_infinity=True,
    )

    best_val_cer = float("inf")
    best_state = None

    for epoch in range(1, num_epochs + 1):
        train_stats = run_epoch(
            model=model,
            loader=train_loader,
            ctc_loss_fn=ctc_loss_fn,
            tokenizer=tokenizer,
            device=device,
            optimizer=optimizer,
            grad_clip_norm=grad_clip_norm,
        )

        with torch.no_grad():
            val_stats = run_epoch(
                model=model,
                loader=val_loader,
                ctc_loss_fn=ctc_loss_fn,
                tokenizer=tokenizer,
                device=device,
                optimizer=None,
            )

        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_stats.loss:.4f} "
            f"train_CER={train_stats.cer:.4f} "
            f"train_WER={train_stats.wer:.4f} || "
            f"val_loss={val_stats.loss:.4f} "
            f"val_CER={val_stats.cer:.4f} "
            f"val_WER={val_stats.wer:.4f}"
        )

        if val_stats.cer < best_val_cer:
            best_val_cer = val_stats.cer
            best_state = {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
                "val_cer": val_stats.cer,
                "val_wer": val_stats.wer,
            }

    return best_state


# -----------------------------
# 8. Example usage
# -----------------------------
def make_dataloader(dataset, tokenizer, batch_size=16, shuffle=True, num_workers=0):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=lambda batch: ctc_collate_fn(batch, tokenizer),
    )

In [ ]:
class NormBetaDataset(Dataset):
    """Simple utterance-level dataset built from norm_beta and utterance bounds."""
    def __init__(self, norm_beta: np.ndarray, utterance_bounds, texts=None):
        self.norm_beta = norm_beta
        self.utterance_bounds = list(utterance_bounds)
        if texts is None:
            self.texts = [""] * len(self.utterance_bounds)
        else:
            if len(texts) != len(self.utterance_bounds):
                raise ValueError("texts must match utterance_bounds length")
            self.texts = list(texts)

    def __len__(self):
        return len(self.utterance_bounds)

    def __getitem__(self, idx):
        start, end = self.utterance_bounds[idx]
        x = torch.as_tensor(self.norm_beta[start:end, :], dtype=torch.float32)
        return {
            "features": x,
            "text": self.texts[idx],
        }


data_length = 10
train_split =int(data_length*0.8)
train_bounds = utterance_bounds[:train_split]
train_text = texts[:train_split]
val_bounds = utterance_bounds[train_split:]
val_text = texts[train_split:]
train_dataset = NormBetaDataset(norm_patches, train_bounds, train_text)
val_dataset = NormBetaDataset(norm_patches, val_bounds, val_text)


tokenizer = CharTokenizer()
train_loader = make_dataloader(train_dataset, tokenizer, batch_size=5, shuffle=True)
val_loader = make_dataloader(val_dataset, tokenizer, batch_size=5, shuffle=False)

# Infer input dimension from one sample
sample = train_dataset[0]["features"]
d_in = sample.shape[1]

model = SMTCTCLinear(
    d_in=d_in,
    vocab_size=tokenizer.vocab_size,
    dropout=0.1,
    use_layernorm=True,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_state = train_ctc_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer=tokenizer,
    device=device,
    num_epochs=3000,
    lr=1e-3,
    weight_decay=1e-4,
    grad_clip_norm=5.0,
)

if best_state is not None:
    torch.save(best_state, "best_smt_ctc_linear.pt")
    print(f"Saved best checkpoint with val CER={best_state['val_cer']:.4f}")

Epoch 001 | train_loss=21.0243 train_CER=4.4445 train_WER=1.0931 || val_loss=19.3682 val_CER=3.3378 val_WER=1.0275
Epoch 002 | train_loss=20.0728 train_CER=4.4020 train_WER=1.2562 || val_loss=18.3063 val_CER=3.4783 val_WER=1.0519
Epoch 003 | train_loss=19.4459 train_CER=4.8138 train_WER=1.3369 || val_loss=17.2920 val_CER=4.0936 val_WER=1.0833
Epoch 004 | train_loss=18.2408 train_CER=5.4503 train_WER=1.7603 || val_loss=16.3699 val_CER=3.8854 val_WER=1.1149
Epoch 005 | train_loss=17.6405 train_CER=4.9833 train_WER=1.9730 || val_loss=15.6135 val_CER=3.7882 val_WER=1.1517
Epoch 006 | train_loss=16.8982 train_CER=4.8121 train_WER=2.1661 || val_loss=15.0376 val_CER=4.2667 val_WER=1.1771
Epoch 007 | train_loss=16.1395 train_CER=4.6327 train_WER=2.4090 || val_loss=14.6144 val_CER=3.4361 val_WER=1.2151
Epoch 008 | train_loss=15.7718 train_CER=4.4525 train_WER=2.7013 || val_loss=14.3087 val_CER=3.1724 val_WER=1.2424
Epoch 009 | train_loss=15.2051 train_CER=4.0819 train_WER=2.7426 || val_loss=14.

In [23]:
from typing import List, Dict, Union, Optional
import torch
import torch.nn.functional as F


@torch.no_grad()
def predict_batch(
    model: torch.nn.Module,
    features: torch.Tensor,
    feature_lens: torch.Tensor,
    tokenizer,
    device: torch.device,
    return_log_probs: bool = False,
    return_confidence: bool = False,
) -> Dict[str, Union[List[str], List[List[int]], torch.Tensor, List[float]]]:
    """
    Greedy CTC decoding for a batch.

    Args:
        model:
            Trained SMTCTCLinear model.
        features:
            Tensor of shape (B, T, D)
        feature_lens:
            Tensor of shape (B,) with valid frame counts
        tokenizer:
            Must provide blank_id and decode(ids)
        device:
            cpu or cuda
        return_log_probs:
            If True, include frame-level log_probs (B, T, V) on CPU
        return_confidence:
            If True, include a simple confidence proxy per utterance

    Returns:
        dict with:
            "texts": List[str]
            "token_ids": List[List[int]]
            optionally "log_probs": Tensor (B, T, V)
            optionally "confidence": List[float]
    """
    model.eval()

    features = features.to(device)
    logits = model(features)                     # (B, T, V)
    log_probs = F.log_softmax(logits, dim=-1)   # (B, T, V)
    log_probs_t = log_probs.transpose(0, 1)     # (T, B, V)

    pred_token_ids = ctc_greedy_decode(
        log_probs=log_probs_t.cpu(),
        input_lens=feature_lens.cpu(),
        blank_id=tokenizer.blank_id,
    )
    pred_texts = [tokenizer.decode(ids) for ids in pred_token_ids]

    out = {
        "texts": pred_texts,
        "token_ids": pred_token_ids,
    }

    if return_confidence:
        # Simple confidence proxy:
        # average max posterior over valid frames
        probs = log_probs.exp()  # (B, T, V)
        confs = []
        for b in range(probs.shape[0]):
            T = int(feature_lens[b].item())
            if T == 0:
                confs.append(0.0)
                continue
            max_post = probs[b, :T].max(dim=-1).values
            confs.append(float(max_post.mean().item()))
        out["confidence"] = confs

    if return_log_probs:
        out["log_probs"] = log_probs.cpu()

    return out


@torch.no_grad()
def predict_one(
    model: torch.nn.Module,
    features: torch.Tensor,
    tokenizer,
    device: torch.device,
    return_log_probs: bool = False,
    return_confidence: bool = False,
) -> Dict[str, Union[str, List[int], torch.Tensor, float]]:
    """
    Greedy CTC decoding for a single utterance.

    Args:
        features:
            Tensor of shape (T, D)

    Returns:
        dict with:
            "text": str
            "token_ids": List[int]
            optionally "log_probs": Tensor (T, V)
            optionally "confidence": float
    """
    if features.ndim != 2:
        raise ValueError(f"Expected features with shape (T, D), got {tuple(features.shape)}")

    batch_features = features.unsqueeze(0)  # (1, T, D)
    feature_lens = torch.tensor([features.shape[0]], dtype=torch.long)

    batch_out = predict_batch(
        model=model,
        features=batch_features,
        feature_lens=feature_lens,
        tokenizer=tokenizer,
        device=device,
        return_log_probs=return_log_probs,
        return_confidence=return_confidence,
    )

    out = {
        "text": batch_out["texts"][0],
        "token_ids": batch_out["token_ids"][0],
    }

    if return_confidence:
        out["confidence"] = batch_out["confidence"][0]

    if return_log_probs:
        out["log_probs"] = batch_out["log_probs"][0]  # (T, V)

    return out

In [13]:
def pad_feature_list(feature_list: List[torch.Tensor]) -> Dict[str, torch.Tensor]:
    """
    Pads a list of (T_i, D) tensors into (B, T_max, D).
    """
    if len(feature_list) == 0:
        raise ValueError("feature_list must be non-empty")

    d_in = feature_list[0].shape[1]
    lengths = [x.shape[0] for x in feature_list]
    max_t = max(lengths)

    batch = torch.zeros(len(feature_list), max_t, d_in, dtype=torch.float32)
    for i, x in enumerate(feature_list):
        if x.ndim != 2:
            raise ValueError(f"Each feature tensor must have shape (T, D), got {tuple(x.shape)}")
        if x.shape[1] != d_in:
            raise ValueError("All feature tensors must have the same feature dimension D")
        batch[i, :x.shape[0]] = x

    return {
        "features": batch,
        "feature_lens": torch.tensor(lengths, dtype=torch.long),
    }


@torch.no_grad()
def predict_feature_list(
    model: torch.nn.Module,
    feature_list: List[torch.Tensor],
    tokenizer,
    device: torch.device,
    return_log_probs: bool = False,
    return_confidence: bool = False,
) -> Dict[str, Union[List[str], List[List[int]], torch.Tensor, List[float]]]:
    """
    Decode a list of variable-length SMT embedding sequences.
    """
    batch = pad_feature_list(feature_list)
    return predict_batch(
        model=model,
        features=batch["features"],
        feature_lens=batch["feature_lens"],
        tokenizer=tokenizer,
        device=device,
        return_log_probs=return_log_probs,
        return_confidence=return_confidence,
    )

In [14]:
def load_trained_model(
    checkpoint_path: str,
    d_in: int,
    vocab_size: int,
    device: torch.device,
    dropout: float = 0.1,
    use_layernorm: bool = True,
):
    model = SMTCTCLinear(
        d_in=d_in,
        vocab_size=vocab_size,
        dropout=dropout,
        use_layernorm=use_layernorm,
    )
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    model.to(device)
    model.eval()
    return model


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = CharTokenizer()

model = load_trained_model(
    checkpoint_path="best_smt_ctc_linear.pt",
    d_in=d_in,
    vocab_size=tokenizer.vocab_size,
    device=device,
)

for sample in train_dataset:


    pred = predict_one(
        model=model,
        features=sample["features"],
        tokenizer=tokenizer,
        device=device,
        return_confidence=True,
    )

    print("Reference :", sample["text"])
    print("Predicted :", pred["text"])
    print("Confidence:", pred["confidence"])

Reference : mister quilter is the apostle of the middle classes and we are glad to welcome his gospel
Predicted : saossso s sos
Confidence: 0.8949673175811768
Reference : nor is mister quilters manner less interesting than his matter
Predicted : osss 
Confidence: 0.9054639339447021
Reference : he tells us that at this festive season of the year with christmas and roast beef looming before us similes drawn from eating and its results occur most readily to the mind
Predicted :   ssss ssss s ssosss s saesn
Confidence: 0.8984240889549255
Reference : he has grave doubts whether sir frederick leightons work is really greek after all and can discover in it but little of rocky ithaca
Predicted :  s   soee ssoe s
Confidence: 0.9096879959106445
Reference : linnells pictures are a sort of up guards and at em paintings and masons exquisite idylls are as national as a jingo poem mister birket fosters landscapes smile at one much in the same way that mister carker used to flash his teeth and mister 